## Import and format the data to build a DataLoader

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import lightning as L
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

In [2]:
url =  Path.cwd()/ "iris.txt"
df = pd.read_table(url, sep=",", header=None)

df.head()

,0,1,2,3,4
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
df.columns = ["sepal_length",
              "sepal_width",
              "petal_length",
              "petal_width",
              "class"]




input_values = df[['petal_width', 'sepal_width']]
label_values = df['class']
class_as_numbers = label_values.factorize()[0]


input_train, input_test, label_train, label_test = train_test_split(input_values,
                                                                    class_as_numbers,
                                                                    test_size=0.25,
                                                                    stratify=class_as_numbers,
                                                                    random_state=42)


one_hot_label_train = F.one_hot(torch.tensor(label_train)).type(torch.float32)

max_vals_in_input_train = input_train.max()
min_vals_in_input_train = input_train.min()


#  Next we normalize input_train adn input_test with max and min values from input_train
input_train = (input_train - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_test = (input_test - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)


input_train_tensors = torch.tensor(input_train.values).type(torch.float32)
input_test_tensors = torch.tensor(input_test.values).type(torch.float32)

train_dataset = TensorDataset(input_train_tensors, one_hot_label_train)
train_dataloader = DataLoader(train_dataset)

## Building the Neural Network

In [4]:
class myNN(L.LightningModule):
    def __init__(self):
        super().__init__()        
        L.seed_everything(seed=42)

        self.input_to_hidden = nn.Linear(in_features=2, out_features=2, bias=True)
        self.hidden_to_output = nn.Linear(in_features=2, out_features=3, bias=True)
        self.loss = nn.CrossEntropyLoss() # applies a SoftMax function to the valies we give it, so we don't have to add SoftMax ourselves

    def forward(self, input):
        hidden = self.input_to_hidden(input)
        output_values = self.hidden_to_output(torch.relu(hidden))
        return output_values

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.001)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.forward(inputs)
        loss = self.loss(outputs, labels)
        return loss


In [5]:
model = myNN()

Seed set to 42


In [6]:
trainer = L.Trainer(max_epochs=10)
trainer.fit(model, train_dataloaders=train_dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enabl

Epoch 9: 100%|██████████| 112/112 [00:00<00:00, 471.07it/s, v_num=6]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 112/112 [00:00<00:00, 461.99it/s, v_num=6]


In [7]:
predictions = model(input_test_tensors)
predicted_labels = torch.argmax(predictions, dim=1)
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

tensor(0.6579)

In [8]:
# 0.65 is not good accuracy so we will continue training
path_to_checkpoint = trainer.checkpoint_callback.best_model_path
trainer = L.Trainer(max_epochs=100)
trainer.fit(model, train_dataloaders=train_dataloader, ckpt_path = path_to_checkpoint)

predictions = model(input_test_tensors)
predicted_labels = torch.argmax(predictions, dim=1)
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Restoring states from the checkpoint path at /Users/odysseasgeorgiades/Desktop/Neural Networks/04/lightning_logs/version_6/checkpoints/epoch=9-step=1120.ckpt
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/Users/odysseasgeorgiades/Desktop/Neural Networks/04/lightning_logs/version_6/checkpoints' to '/Users/odysseasgeorgiades/Desktop/Neural Networks/04/lightning_lo

Epoch 99: 100%|██████████| 112/112 [00:00<00:00, 497.12it/s, v_num=7]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 112/112 [00:00<00:00, 487.55it/s, v_num=7]


tensor(0.9474)

In [9]:
# We achieved 94.74% accuracy
# We print all the weights and biases.
for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=2))

input_to_hidden.weight tensor([[ 3.5500,  0.2500],
        [-1.9600,  1.4800]])
input_to_hidden.bias tensor([-0.3400,  1.2400])
hidden_to_output.weight tensor([[-4.2000,  3.1500],
        [ 0.1600,  0.3300],
        [ 1.8700, -3.2900]])
hidden_to_output.bias tensor([ 0.4600,  0.9600, -0.6100])


## Making a prediction

In [10]:
input_sample = pd.Series([0.2, 3.0], index=min_vals_in_input_train.index)
normalized_values = (input_sample - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
tensor_input = torch.tensor(normalized_values.values, dtype=torch.float32).unsqueeze(0)

with torch.no_grad():
    output = model(tensor_input)
    predicted_class = torch.argmax(output, dim=1)
    print(predicted_class.tolist())


[0]


We predicted 0. That means it is Setosa